In [2]:
import json
import math # Import math for handling potential non-finite numbers if needed, though float() handles most cases.

# --- User-provided Helper Functions (with minor robustness improvements) ---

def check_overlap(start1, end1, start2, end2):
    """Checks if two ranges [start1, end1] and [start2, end2] overlap."""
    # Ensure values are numbers for comparison
    try:
        start1, end1, start2, end2 = float(start1), float(end1), float(start2), float(end2)
    except (ValueError, TypeError):
        # print(f"Warning: Non-numeric range value encountered. Overlap check may fail: {start1}, {end1}, {start2}, {end2}")
        return False # Cannot determine overlap if values are not numeric

    # Check for invalid ranges (start > end) - consider them non-overlapping for safety
    if start1 > end1 or start2 > end2:
        return False

    # Standard overlap check: max(start) <= min(end)
    return max(start1, start2) <= min(end1, end2)

def are_partially_agreed(sense1, sense2):
    """
    Checks if two sense objects meet the partial agreement criteria:
    1) The "sense" has to be exactly the same.
    2) Both Arg1_start to Arg1_end and Arg2_start to Arg2_end have to overlap.
    """
    # Basic structure check - ensure required keys exist
    required_keys = ["sense", "Arg1_start", "Arg1_end", "Arg2_start", "Arg2_end"]
    for key in required_keys:
        if key not in sense1 or key not in sense2:
            # print(f"Warning: Missing required key '{key}' in one of the senses for agreement check.")
            return False # Cannot check agreement without required keys

    # Condition 1: Sense must be the same
    if sense1["sense"] != sense2["sense"]:
        return False

    # Condition 2: Arg1 ranges must overlap
    arg1_overlap = check_overlap(sense1["Arg1_start"], sense1["Arg1_end"],
                                 sense2["Arg1_start"], sense2["Arg1_end"])

    # Condition 3: Arg2 ranges must overlap
    arg2_overlap = check_overlap(sense1["Arg2_start"], sense1["Arg2_end"],
                                 sense2["Arg2_start"], sense2["Arg2_end"])

    return arg1_overlap and arg2_overlap

def merge_sense_objects(sense1, sense2):
    """Merges two partially agreed sense objects."""
    # Ensure required keys exist before merging
    required_keys = ["sense", "Arg1_start", "Arg1_end", "Arg2_start", "Arg2_end", "explicit", "confidence"]
    for key in required_keys:
         if key not in sense1 or key not in sense2:
             print(f"Error: Attempted to merge senses missing required key '{key}'. Returning sense1.")
             return sense1 # Or raise an error, or return None

    # Safely convert ranges to float for min/max calculation
    try:
        arg1_start1, arg1_end1 = float(sense1["Arg1_start"]), float(sense1["Arg1_end"])
        arg2_start1, arg2_end1 = float(sense1["Arg2_start"]), float(sense1["Arg2_end"])
        arg1_start2, arg1_end2 = float(sense2["Arg1_start"]), float(sense2["Arg1_end"])
        arg2_start2, arg2_end2 = float(sense2["Arg2_start"]), float(sense2["Arg2_end"])
    except (ValueError, TypeError):
         print(f"Error converting range values to float during merge. Returning sense1.")
         return sense1 # Cannot merge if ranges are not numeric

    merged_sense = {
        "sense": sense1["sense"], # Sense is guaranteed to be the same by are_partially_agreed
        "Arg1_start": min(arg1_start1, arg1_start2),
        "Arg1_end": max(arg1_end1, arg1_end2),
        "Arg2_start": min(arg2_start1, arg2_start2),
        "Arg2_end": max(arg2_end1, arg2_end2),
    }

    # Merge explicit values - handle potential missing 'explicit' key gracefully
    explicit1 = sense1.get("explicit", "") # Use .get with default empty string
    explicit2 = sense2.get("explicit", "")

    if explicit1 == explicit2:
        merged_sense["explicit"] = explicit1
    elif explicit1 == "implicit":
        merged_sense["explicit"] = explicit2
    elif explicit2 == "implicit":
        merged_sense["explicit"] = explicit1
    else:
        explicits1_parts = explicit1.split(' | ') if explicit1 else []
        explicits2_parts = explicit2.split(' | ') if explicit2 else []
        combined_explicits = sorted(list(set(explicits1_parts + explicits2_parts)))
        merged_sense["explicit"] = " | ".join(combined_explicits)

    # Merge confidence - take the maximum confidence of the merged senses
    # Handle potential missing or non-numeric confidence
    try:
        conf1 = float(sense1.get("confidence", 0.0)) # Default to 0.0 if missing/invalid
    except (ValueError, TypeError):
         conf1 = 0.0 # Use 0 if conversion fails
    try:
        conf2 = float(sense2.get("confidence", 0.0))
    except (ValueError, TypeError):
         conf2 = 0.0

    merged_sense["confidence"] = max(conf1, conf2)

    return merged_sense

# --- Main Processing Logic ---

def process_line(json_obj):
    """
    Processes a single JSON object (representing one document/entry)
    to remove duplicates, merge partially agreed senses, and resolve
    same-range different-sense conflicts by keeping the highest confidence.
    """
    # 1. Remove exact duplicate "senses" objects
    senses = json_obj.get("Senses", [])
    seen_senses_str = set()
    unique_senses = []
    for sense in senses:
        # Convert to hashable string representation (sorted keys for consistency)
        try:
            sense_str = json.dumps(sense, sort_keys=True)
            if sense_str not in seen_senses_str:
                seen_senses_str.add(sense_str)
                unique_senses.append(sense)
        except TypeError as e:
            print(f"Warning: Could not serialize sense object for deduplication: {sense}. Skipping. Error: {e}")
            # Optionally, you could try to process this sense without deduplication against others,
            # but skipping it is safer if its structure is unexpected.


    # 2. Merge partial agreed "senses" objects (iterative process)
    current_senses = unique_senses
    merged_occurred = True

    # Keep merging as long as any merges happen in a pass
    while merged_occurred:
        merged_occurred = False
        next_senses_list = []
        used_indices = set() # Track indices of senses that have been merged into others

        for i in range(len(current_senses)):
            # If this sense has already been merged into another in this pass, skip it
            if i in used_indices:
                continue

            # Start with the current sense as the base for potential merges in this pass
            base_sense = current_senses[i]
            current_merged_sense = base_sense
            used_indices.add(i) # Mark the base sense as 'used' (either kept as is or merged)

            # Check against all subsequent senses that haven't been used yet
            for j in range(i + 1, len(current_senses)):
                if j in used_indices:
                    continue

                # Check if the *current state* of our merged sense can merge with sense j
                if are_partially_agreed(current_merged_sense, current_senses[j]):
                    # Merge them, update the current_merged_sense
                    current_merged_sense = merge_sense_objects(current_merged_sense, current_senses[j])
                    used_indices.add(j) # Mark sense j as having been merged into the base sense i
                    merged_occurred = True # A merge happened in this pass

            # Add the final result of merging for this base sense to the list for the next pass
            next_senses_list.append(current_merged_sense)

        # Update the list of senses for the next iteration
        current_senses = next_senses_list

    # 3. For "sense" objects that have identical arg1 and arg2 but different senses,
    #    keep the one with the largest confidence level.

    # Group senses by their Arg1 and Arg2 ranges
    grouped_by_range = {}
    for sense in current_senses:
        # Create a unique, hashable key based on the range
        # Use .get and handle potential non-numeric/missing values for safety
        try:
            arg1_start = float(sense.get("Arg1_start", math.nan))
            arg1_end = float(sense.get("Arg1_end", math.nan))
            arg2_start = float(sense.get("Arg2_start", math.nan))
            arg2_end = float(sense.get("Arg2_end", math.nan))
            # Use a string representation robust to float variations if needed,
            # but direct floats in a tuple are usually fine unless precision issues arise.
            # Using string representation is safer for exact match:
            range_key = f"{arg1_start}-{arg1_end}_{arg2_start}-{arg2_end}"
        except (ValueError, TypeError):
             # If range values are invalid, group them under a special key or skip
             range_key = "invalid_range"
             print(f"Warning: Invalid range values encountered during grouping: {sense}. Grouped under '{range_key}'.")

        if range_key not in grouped_by_range:
            grouped_by_range[range_key] = []
        grouped_by_range[range_key].append(sense)

    final_senses = []
    # Process each group (senses sharing the exact same range)
    for range_key, senses_list in grouped_by_range.items():
        if range_key == "invalid_range":
            # Optionally process invalid ranges differently or skip
            final_senses.extend(senses_list) # Just add invalid ones back without further processing
            continue

        # Sort the senses within this group by confidence level in descending order
        # Handle potential missing or non-numeric confidence values for sorting
        # Default to a very low number (-1.0) for sorting if confidence is missing or invalid
        try:
            sorted_senses = sorted(senses_list, key=lambda s: float(s.get("confidence", -1.0)), reverse=True)
            # Keep only the sense with the highest confidence for this specific range
            if sorted_senses: # Ensure the list is not empty
                 final_senses.append(sorted_senses[0])
        except Exception as e:
            print(f"Error sorting senses by confidence for range {range_key}. Adding all senses in this group. Error: {e}")
            final_senses.extend(senses_list) # Fallback: add all senses if sorting fails


    # Prepare the final output object with just 'id' and the processed 'Senses'
    return {"id": json_obj.get("id"), "Senses": final_senses}

def process_jsonl_file(input_filepath, output_filepath):
    """
    Reads a JSONL file, processes each line using process_line,
    and writes the results to a new JSONL file.
    """
    print(f"Processing started for input file: {input_filepath}")
    processed_count = 0
    error_count = 0

    # Open input and output files
    with open(input_filepath, 'r', encoding='utf-8') as infile, \
         open(output_filepath, 'w', encoding='utf-8') as outfile:

        # Process each line in the input file
        for line_num, line in enumerate(infile, 1):
            line = line.strip()
            if not line: # Skip empty lines
                continue

            try:
                # Load the JSON object from the line
                json_obj = json.loads(line)

                # Process the JSON object
                processed_obj = process_line(json_obj)

                # Write the processed object to the output file as a JSONL line
                outfile.write(json.dumps(processed_obj) + '\n')
                processed_count += 1

            except json.JSONDecodeError as e:
                print(f"Error decoding JSON on line {line_num}: {line[:100]}... Error: {e}")
                error_count += 1
            except Exception as e:
                print(f"An unexpected error occurred processing line {line_num}: {line[:100]}... Error: {e}")
                error_count += 1

    print(f"Processing finished.")
    print(f"Successfully processed {processed_count} lines.")
    if error_count > 0:
        print(f"Encountered errors on {error_count} lines.")

# --- Example Usage in IPython Notebook ---


input_jsonl_path = "/content/data/output_merged_explicit_claude.jsonl"
output_jsonl_path = "/content/data/(no_uplicate)output_merged_explicit_claude.jsonl"

# Run the processing function
process_jsonl_file(input_jsonl_path, output_jsonl_path)

# Display the content of the output file in IPython Notebook
print("\n--- Output JSONL ---")
with open(output_jsonl_path, "r", encoding="utf-8") as f:
    print(f.read())

# Optional: Clean up dummy files (useful if running multiple times)
# import os
# if os.path.exists(input_jsonl_path):
#     os.remove(input_jsonl_path)
# if os.path.exists(output_jsonl_path):
#     os.remove(output_jsonl_path)

Processing started for input file: /content/data/output_merged_explicit_claude.jsonl
Processing finished.
Successfully processed 12 lines.

--- Output JSONL ---
{"id": "How_to_dig_basement_spansection", "Senses": [{"Arg1_start": 7.0, "Arg1_end": 7.0, "Arg2_start": 9.0, "Arg2_end": 9.0, "sense": "Comparison.Concession", "explicit": "but", "confidence": 0.9}, {"Arg1_start": 12.0, "Arg1_end": 12.0, "Arg2_start": 13.0, "Arg2_end": 13.0, "sense": "Expansion.Level-of-Detail", "explicit": "now", "confidence": 0.85}, {"sense": "Contingency.Cause", "Arg1_start": 16.0, "Arg1_end": 16.0, "Arg2_start": 17.0, "Arg2_end": 17.0, "explicit": "And | and", "confidence": 0.9}, {"sense": "Temporal.Asynchronous", "Arg1_start": 17.0, "Arg1_end": 17.0, "Arg2_start": 18.0, "Arg2_end": 18.0, "explicit": "Now | now", "confidence": 0.95}, {"Arg1_start": 18.0, "Arg1_end": 18.0, "Arg2_start": 19.0, "Arg2_end": 19.0, "sense": "Expansion.Level-of-Detail", "explicit": "in", "confidence": 0.7}, {"Arg1_start": 26.0, "A